In [ ]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer
import math

# Text Tokenization
- Word-level: Simple, high OOV (out of vocabulary) and large vocab
- Character-level: No OOV but very long sequence
- Subword: practical, balances

In [1]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
input = "Hello, how are you?"
tokens = tokenizer(input, return_tensors="pt")
print(tokens)


/Users/pengu/Developer/Deep Learning PyTorch Codebook/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'input_ids': tensor([[ 101, 7592, 1010, 2129, 2024, 2017, 1029,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]])}


# Image/Audio/Video Tokenization
- Patch/chunk tokenization is simple & parallel
- Too small token unit -> Long n (costly attention)
- WARNING: preprocessing need to match train & inference

In [ ]:
# Image Patch Tokenizer (ViT style)
class PatchTokenizer(nn.Module):
    def __init__(self, patch_size=16, in_channels=3, embed_dim=768):
        super(PatchTokenizer, self).__init__()
        self.conv = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)
    
    def forward(self, x):
        x = self.conv(x)  # (B, embed_dim, H/patch_size, W/patch_size)
        x = x.flatten(2)  # (B, embed_dim, N_patches)
        x = x.transpose(1, 2)  # (B, N_patches, embed_dim)
        return x

In [ ]:
# Audio Frame Tokenizer
# frontend: waveform -> frames features (log-mel spectrogram, MFCC, etc.)
class AudioTokenizer(nn.Module):
    def __init__(self, in_features=128, embed_dim=768):
        super(AudioTokenizer, self).__init__()
        self.linear = nn.Linear(in_features, embed_dim)
    
    def forward(self, x):
        x = self.linear(x)  # (B, T, embed_dim)
        return x

In [ ]:
# Video Tubelet Tokenizer
# tubelet: 3D patch (T, H, W)
class TubeletTokenizer(nn.Module):
    def __init__(self, tubelet_size=(2, 16, 16), in_channels=3, embed_dim=768):
        super(TubeletTokenizer, self).__init__()
        self.conv = nn.Conv3d(in_channels, embed_dim, kernel_size=tubelet_size, stride=tubelet_size)
    
    def forward(self, x):
        x = self.conv(x)  # (B, embed_dim, T/tubelet_T, H/tubelet_H, W/tubelet_W)
        x = x.flatten(2)  # (B, embed_dim, N_tubelets)
        x = x.transpose(1, 2)  # (B, N_tubelets, embed_dim)
        return x

# Token vs Positional Embedding
- Token embedding: learned lookup, pretrained/fronzen/finetuned, tied input/output embedding
- Positional embedding: sinusoidal/learned table, bias (token distance), rotary (RoPE) (rotate Q/K by position)
- Input: E_token + E_pos = X_0

In [ ]:
# Learned/Tied Token Embedding
tok_embed = nn.Embedding(num_embeddings=10000, embedding_dim=768)
lm_head = nn.Linear(768, 10000)
# Tying weights
lm_head.weight = tok_embed.weight
# Pad ID fixed, mask padded tokens in loss/attention
# Rebuild entire embedding matrix if vocab changes, no OOV token, etc.

# Positional Encoding
- Give order awareness (attention gives global interaction)
- Permutation inequivariant
- Shape:
+ Token: `B x n x d` (n: vocab size, d: dimension)
+ Position: `1 x n x d`
## Absolute Positional Encoding
### Sinusoidal Positional Encoding
- Fixed: `PE(pos, 2i) = sin(pos/10000^(2i/d)), PE(pos, 2i + 1) = cos(pos/10000^(2i/d))`
- Simple, no parameters
- Same table can be generated, for any sequence length
- Different dimensions encode different frequencies
- Use multi-frequency periodic signals

In [ ]:
def sinusoidal_positional_encoding(seq_len, embed_dim):
    position = torch.arange(seq_len).unsqueeze(1)  # (seq_len, 1)
    div_term = torch.exp(torch.arange(0, embed_dim, 2) * -(torch.log(torch.tensor(10000.0)) / embed_dim))  # (embed_dim/2,)
    pe = torch.zeros(seq_len, embed_dim)
    pe[:, 0::2] = torch.sin(position * div_term)  # Even indices
    pe[:, 1::2] = torch.cos(position * div_term)  # Odd indices
    return pe

### Learned absolute position embedding
- Use a trainable lookup table size: `n_max x d` (Parameters: `n_max x d`)
- `X_0 = E_token + E_pos[0:n]`
- Task adaptive, easy to use
- Bound to trained max length

In [ ]:
tok_embed = nn.Embedding(num_embeddings=10000, embedding_dim=768)
pos_embed = sinusoidal_positional_encoding(seq_len=512, embed_dim=768)
# pos_embed is fixed, not learned
x = tok_embed(input_ids) + pos_embed[:input_ids.size(1), :]  # (B, seq_len, embed_dim)

### Relative position bias (Minimal score update)
- Distance aware, transfers well
- Extra bias tables/hyperparameters
- Parameters: depends on range/windows

In [ ]:
# Scores: (B, h, n, n)
# Relative position bias: (h, n, n)
score = (Q @ K.transpose(-2, -1)) / math.sqrt(d_k) + relative_position_bias.unsqueeze(0)  # Broadcast to (B, h, n, n)
attn = torch.softmax(score, dim=-1)  # (B, h, n, n)

### RoPE
- Injects position into Q/K geometry (rotate Q/K) before score computation
- Strong long-context behavior
- Complex implementation detaols

In [ ]:
# q, k: (B, h, n, d_k)
q_rot = rope(q, position_ids)
k_rot = rope(k, position_ids)

scores = (q_rot @ k_rot.transpose(-2, -1)) / math.sqrt(d_k)
attn = torch.softmax(scores, dim=-1)

## Relative positional information
- token i after token j -> distance `i - j`
- Add relative position bias (b_i - j) to attention scores
- Modify attention score directly
### RoPE
- Add relation position via dot product (R_iq_i)^T(R_jk_j)
# Masking
- Pad tokens -> Need to mask to avoid fef